In [1]:
import gzip
import dpkt 
import struct
from collections import defaultdict
import pandas as pd
import os
import pyarrow as pa
import pyarrow.parquet as pq
import glob
from pathlib import Path
import time
import json
import os
import re

In [10]:
gz_path = r"C:\IEX\cache\data_feeds*.pcap.gz"

volume_by_symbol = defaultdict(int)
trades_by_symbol = defaultdict(int)

file_path = glob.glob(gz_path)[0]

In [3]:
big_names = ['NU','GRAB', 'NVDA', 'AGNC', 'INTC', 'RIG', 'NFLX','NOK', 'CLF', 'KO']
print(len(big_names))

10


In [4]:
big_names = set(big_names)

output_dir = r"C:\IEX\20261009"
os.makedirs(output_dir, exist_ok=True)

writers = {}
buffers = {}

columns = [
    "timestamp",
    "symbol",
    "price",
    "size",
    "trade_id",
    "flags",
    "packet_no"
]

In [5]:
def get_writer(symbol):

    if symbol not in writers:
        schema = pa.schema([
            ("timestamp", pa.int64()),
            ("symbol", pa.string()),
            ("price", pa.float64()),
            ("size", pa.int64()),
            ("trade_id", pa.int64()),
            ("flags", pa.int64()),
            ("packet_no", pa.int64()),
        ])

        path = os.path.join(output_dir, f"{symbol}_trades.parquet")
        writers[symbol] = pq.ParquetWriter(path, schema, compression="snappy")
        buffers[symbol] = []

    return writers[symbol]

In [ ]:
BUFFER_SIZE = 100_000
buffers = {symbol: [] for symbol in big_names}

with gzip.open(file_path, "rb") as f:
    pcap = dpkt.pcapng.Reader(f)
    for packet_no, (pcap_ts, raw_packet) in enumerate(pcap):
        eth = dpkt.ethernet.Ethernet(raw_packet)
        ip = eth.data
        if not isinstance(ip, dpkt.ip.IP):
            continue
        udp = ip.data
        if not isinstance(udp, dpkt.udp.UDP):
            continue
        payload = udp.data

        message_count = struct.unpack_from( "<H", payload, 14)[0]
        data = payload[40:]
        pos = 0

        for _ in range(message_count):
            if pos + 2 > len(data):
                break

            msg_len = struct.unpack_from("<H", data, pos)[0]
            msg_start = pos + 2
            msg_end = msg_start + msg_len
            if msg_end > len(data):
                break

            msg = data[msg_start:msg_end]
            if len(msg) >= 38 and msg[0] == ord("T"):

                symbol = msg[10:18].decode(
                    "ascii",
                    errors="ignore"
                ).rstrip("\x00").strip()

                if symbol in big_names:
                    timestamp = struct.unpack_from("<Q", msg, 2)[0]
                    flags = msg[1]
                    size = struct.unpack_from("<I", msg, 18)[0]
                    price = struct.unpack_from("<q", msg, 22)[0] / 10000.0
                    trade_id = struct.unpack_from("<Q", msg, 30)[0]
                    buffers[symbol].append({
                        "timestamp": timestamp,
                        "symbol": symbol,
                        "price": price,
                        "size": size,
                        "trade_id": trade_id,
                        "flags": flags,
                        "packet_no": packet_no
                    })

                    # Write every 100k trades
                    if len(buffers[symbol]) >= BUFFER_SIZE:

                        table = pa.Table.from_pylist(buffers[symbol])
                        get_writer(symbol).write_table(table)
                        buffers[symbol].clear()

        if packet_no % 10_000_000 == 0:
            print(f"Processed {packet_no:,} packets")